# BioSentinel — Longitudinal Biomarker Trend Analysis Demo

This notebook demonstrates how BioSentinel processes a patient's 3-year health timeline
and generates disease risk predictions with explainability.

**All data used in this notebook is fully synthetic.**

---
Author: Mohit Chaprana / Liveupx Pvt. Ltd.  
Repository: https://github.com/liveupx/biosentinel

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json
from pathlib import Path

plt.style.use('dark_background')
plt.rcParams['figure.facecolor'] = '#030c14'
plt.rcParams['axes.facecolor'] = '#071526'
plt.rcParams['axes.edgecolor'] = '#1a3a2a'
plt.rcParams['text.color'] = '#e8f4f0'
plt.rcParams['axes.labelcolor'] = '#8fb3a0'
plt.rcParams['xtick.color'] = '#4a7060'
plt.rcParams['ytick.color'] = '#4a7060'
plt.rcParams['grid.color'] = '#1a3a2a'
plt.rcParams['grid.alpha'] = 0.5

print('✅ BioSentinel Demo Notebook Ready')

In [ ]:
# Load synthetic patient data
with open('../data/samples/patient_sample.json') as f:
    patient = json.load(f)

print(f"Patient: {patient['patient_id']}")
print(f"Age: {patient['demographics']['age']}, Sex: {patient['demographics']['sex']}")
print(f"Checkups available: {len(patient['checkups'])}")
print(f"Monitoring period: {patient['checkups'][0]['date']} → {patient['checkups'][-1]['date']}")

In [ ]:
# Build longitudinal biomarker dataframe
rows = []
for chk in patient['checkups']:
    row = {'date': pd.to_datetime(chk['date'])}
    # Flatten lab results
    for category, labs in chk.get('lab_results', {}).items():
        if isinstance(labs, dict):
            for k, v in labs.items():
                if isinstance(v, (int, float)):
                    row[k] = v
    # Add vitals
    for k, v in chk.get('vitals', {}).items():
        if isinstance(v, (int, float)):
            row[k] = v
    rows.append(row)

df = pd.DataFrame(rows).set_index('date').sort_index()
print(f'Timeline shape: {df.shape}')
print(f'Biomarkers tracked: {df.shape[1]}')
df.head()

In [ ]:
# Plot key biomarker trajectories
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('BioSentinel — Patient Biomarker Trends (3-Year Window)', 
             fontsize=14, fontweight='bold', color='#e8f4f0', y=1.02)

biomarkers = [
    ('hba1c', 'HbA1c (%)', '#00e896', (5.0, 6.5), 'Pre-diabetes range: 5.7-6.4%'),
    ('hemoglobin', 'Hemoglobin (g/dL)', '#00d4e8', (12.0, 16.0), 'Normal: 12.0-16.0 g/dL (F)'),
    ('lymphocytes_pct', 'Lymphocytes (%)', '#f8c25b', (20, 40), 'Normal: 20-40%'),
    ('cea', 'CEA (ng/mL)', '#ff8c42', (0, 5), 'Normal: <5.0 ng/mL'),
    ('alt', 'ALT (U/L)', '#ae81ff', (7, 40), 'Normal: 7-40 U/L (F)'),
    ('glucose_fasting', 'Fasting Glucose (mg/dL)', '#ff4d6d', (70, 100), 'Normal: 70-100 mg/dL'),
]

for ax, (col, label, color, ref_range, ref_note) in zip(axes.flat, biomarkers):
    if col in df.columns:
        values = df[col].dropna()
        dates = values.index
        
        ax.plot(dates, values, marker='o', color=color, linewidth=2, 
                markersize=7, markerfacecolor=color, zorder=5)
        
        # Add trend line
        if len(values) > 2:
            z = np.polyfit(range(len(values)), values.values, 1)
            p = np.poly1d(z)
            ax.plot(dates, p(range(len(values))), '--', 
                    color=color, alpha=0.4, linewidth=1.5, label='Trend')
        
        # Reference range shading
        ax.axhspan(ref_range[0], ref_range[1], alpha=0.08, color='#00e896', zorder=1)
        
        ax.set_title(label, fontsize=10, fontweight='bold', color='#e8f4f0')
        ax.set_xlabel('Date', fontsize=8)
        ax.grid(True, linestyle='--', alpha=0.3)
        ax.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig('../docs/sample_biomarker_trends.png', dpi=150, bbox_inches='tight', 
            facecolor='#030c14')
plt.show()
print('📊 Biomarker trend chart saved')

In [ ]:
# Simple longitudinal trend analysis
print('BioSentinel — Biomarker Trend Analysis Report')
print('=' * 55)

concern_markers = {
    'hba1c': {'normal_max': 5.6, 'unit': '%', 'disease': 'Pre-diabetes'},
    'cea': {'normal_max': 5.0, 'unit': 'ng/mL', 'disease': 'Cancer marker'},
    'alt': {'normal_max': 40, 'unit': 'U/L', 'disease': 'Liver disease'},
    'glucose_fasting': {'normal_max': 100, 'unit': 'mg/dL', 'disease': 'Diabetes risk'},
}

for marker, info in concern_markers.items():
    if marker in df.columns:
        vals = df[marker].dropna().values
        if len(vals) >= 2:
            trend = (vals[-1] - vals[0]) / vals[0] * 100
            latest = vals[-1]
            above_normal = latest > info['normal_max']
            
            status = '🔴 ELEVATED' if above_normal else ('🟡 TRENDING UP' if trend > 5 else '🟢 NORMAL')
            print(f'\n{marker.upper()} ({info["disease"]})')
            print(f'  Latest: {latest:.1f} {info["unit"]} (normal max: {info["normal_max"]})')
            print(f'  24-month trend: {trend:+.1f}%')
            print(f'  Status: {status}')

In [ ]:
# Mock BioSentinel risk score output
# (In production this would be the real model prediction)

risk_results = {
    'Metabolic Risk': {'score': 65, 'level': 'HIGH', 'color': '#ff8c42'},
    'Cancer Risk (composite)': {'score': 22, 'level': 'LOW', 'color': '#00e896'},
    'Cardiovascular Risk': {'score': 42, 'level': 'MODERATE', 'color': '#f8c25b'},
    'Hepatic Risk': {'score': 31, 'level': 'MODERATE', 'color': '#f8c25b'},
    'Hematologic Risk': {'score': 44, 'level': 'MODERATE', 'color': '#f8c25b'},
}

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#030c14')
ax.set_facecolor('#071526')

diseases = list(risk_results.keys())
scores = [risk_results[d]['score'] for d in diseases]
colors = [risk_results[d]['color'] for d in diseases]

bars = ax.barh(diseases, scores, color=colors, alpha=0.85, height=0.55)
ax.axvline(x=25, color='#00e896', linestyle='--', alpha=0.4, linewidth=1)
ax.axvline(x=50, color='#f8c25b', linestyle='--', alpha=0.4, linewidth=1)
ax.axvline(x=75, color='#ff4d6d', linestyle='--', alpha=0.4, linewidth=1)

for bar, score, disease in zip(bars, scores, diseases):
    level = risk_results[disease]['level']
    ax.text(score + 1, bar.get_y() + bar.get_height()/2,
            f'{score}/100  [{level}]', va='center', fontsize=9, 
            color='#e8f4f0', fontweight='bold')

ax.set_xlim(0, 115)
ax.set_xlabel('Risk Score (0–100)', color='#8fb3a0')
ax.set_title('BioSentinel Risk Assessment — Patient p_sample_001 (Jan 2024)', 
             color='#e8f4f0', fontsize=12, fontweight='bold', pad=15)
ax.grid(axis='x', linestyle='--', alpha=0.2)

plt.tight_layout()
plt.savefig('../docs/sample_risk_assessment.png', dpi=150, bbox_inches='tight',
            facecolor='#030c14')
plt.show()
print('\n📊 Risk assessment chart saved')
print('\n⚕️  DISCLAIMER: These are mock risk scores from synthetic data.')
print('   BioSentinel predictions must be reviewed by qualified healthcare professionals.')